## SAM 3 ENTRENADO

In [ ]:
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree
from scipy.spatial.distance import directed_hausdorff
import numpy as np
import cv2

def get_boundary(mask, width=2):
    eroded = binary_erosion(mask, iterations=width)
    return mask & ~eroded

def boundary_iou(pred_mask, gt_mask, width=2):
    pred_boundary = get_boundary(pred_mask, width)
    gt_boundary   = get_boundary(gt_mask, width)
    intersection  = np.logical_and(pred_boundary, gt_boundary).sum()
    union         = np.logical_or(pred_boundary, gt_boundary).sum()
    return intersection / union if union > 0 else 0

def resize_for_hausdorff(pred, gt, max_size=512):
    if max(pred.shape) > max_size:
        scale = max_size / max(pred.shape)
        new_h = int(pred.shape[0] * scale)
        new_w = int(pred.shape[1] * scale)
        pred = cv2.resize(pred.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
        gt   = cv2.resize(gt.astype(np.uint8),   (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
    return pred, gt

def hausdorff_95(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]
    return np.percentile([d1, d2], 95)


In [ ]:
import time
import torch

def measure_inference_sam3_points(model, img_path, central_point):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    start = time.time()
    results = model(img_path, points=central_point, labels=[1], verbose=False)
    latency = (time.time() - start) * 1000
    vram = torch.cuda.memory_allocated() / 1024**2 - vram_before if torch.cuda.is_available() else 0
    return results, latency, vram


In [3]:
import numpy as np

def compute_all_metrics(pred_mask, gt_mask):
    """a"""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f2 = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    return iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy

In [ ]:
import os
import shutil
import random

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"
OUTPUT_ROOT  = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016_split"

TRAIN_IMAGES = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data")
TRAIN_MASKS  = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth")
TEST_IMAGES  = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data")
TEST_MASKS   = os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth")

def split_dataset(train_ratio=0.85):
    train_names = [
        f.replace("_Segmentation.png", "")
        for f in os.listdir(TRAIN_MASKS) if f.endswith("_Segmentation.png")
    ]
    test_names = [
        f.replace("_Segmentation.png", "")
        for f in os.listdir(TEST_MASKS) if f.endswith("_Segmentation.png")
    ]

    random.seed(42)
    random.shuffle(train_names)
    n_train = int(len(train_names) * train_ratio)

    splits = {
        "train": [(n, TRAIN_IMAGES, TRAIN_MASKS) for n in train_names[:n_train]],
        "val":   [(n, TRAIN_IMAGES, TRAIN_MASKS) for n in train_names[n_train:]],
        "test":  [(n, TEST_IMAGES,  TEST_MASKS)  for n in test_names],
    }

    for split, entries in splits.items():
        os.makedirs(os.path.join(OUTPUT_ROOT, "images", split), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_ROOT, "masks",  split), exist_ok=True)
        for name, img_src_dir, mask_src_dir in entries:
            shutil.copy(
                os.path.join(img_src_dir,  name + ".jpg"),
                os.path.join(OUTPUT_ROOT, "images", split, name + ".jpg")
            )
            shutil.copy(
                os.path.join(mask_src_dir, name + "_Segmentation.png"),
                os.path.join(OUTPUT_ROOT, "masks",  split, name + ".png")
            )

split_dataset()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from ultralytics import SAM
from ultralytics.models.sam import SAM3Predictor
import cv2
import numpy as np
import os
import json

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, img_size=1008):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        for img_filename in os.listdir(self.images_dir):
            if not img_filename.lower().endswith(".jpg"):
                continue
            name      = img_filename.replace(".jpg", "")
            img_path  = os.path.join(self.images_dir, img_filename)
            mask_path = os.path.join(self.masks_dir,  name + ".png")
            if os.path.exists(mask_path):
                self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image          = cv2.imread(img_path)
        orig_h, orig_w = image.shape[:2]
        scale_x        = self.img_size / orig_w
        scale_y        = self.img_size / orig_h
        image          = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image          = cv2.resize(image, (self.img_size, self.img_size))
        image          = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask  = cv2.resize(mask, (288, 288))
        mask  = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        gt_full  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_bin   = (gt_full > 127).astype(np.uint8)
        contours, _ = cv2.findContours(gt_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        x, y, w, h  = cv2.boundingRect(np.vstack(contours))

        cx    = (x + w / 2) * scale_x
        cy    = (y + h / 2) * scale_y
        point = torch.tensor([[cx, cy]]).float()
        label = torch.tensor([1])

        return image, mask, point, label


def train_sam(dataset_path, weights_path, output_weights, epochs=50, batch_size=4):
    sam3_wrapper = SAM(weights_path)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sam3.to(device)

    for param in sam3.image_encoder.parameters():
        param.requires_grad = False
    for param in sam3.sam_prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam3.sam_mask_decoder.parameters(), lr=1e-4)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train")
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    predictor = SAM3Predictor(overrides={"model": weights_path, "task": "segment", "mode": "predict", "verbose": False})
    predictor.setup_model(sam3)
    predictor._bb_feat_sizes = [(288, 288), (144, 144), (72, 72)]

    sam3.train()

    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            masks  = masks.to(device)
            points = points.to(device)
            labels = labels.to(device)

            loss_batch = 0
            for i in range(images.shape[0]):
                with torch.no_grad():
                    image_tensor = images[i].unsqueeze(0).to(device)
                    backbone_out = sam3.forward_image(image_tensor)
                    _, vision_feats, _, _ = sam3._prepare_backbone_features(backbone_out)
                    if sam3.directly_add_no_mem_embed:
                        vision_feats[-1] = vision_feats[-1] + sam3.no_mem_embed
                    feats = [
                        feat.permute(1, 2, 0).reshape(1, -1, *feat_size)
                        for feat, feat_size in zip(vision_feats, predictor._bb_feat_sizes)
                    ]
                    features = {"image_embed": feats[-1], "high_res_feats": feats[:-1]}

                sparse_embeddings, dense_embeddings = sam3.sam_prompt_encoder(
                    points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                    boxes=None,
                    masks=None
                )

                image_embedding   = features["image_embed"]
                image_pe          = sam3.sam_prompt_encoder.get_dense_pe()
                high_res_features = features["high_res_feats"]

                low_res_masks, _, _, _ = sam3.sam_mask_decoder(
                    image_embeddings=image_embedding,
                    image_pe=image_pe,
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                    repeat_image=False,
                    high_res_features=high_res_features,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save({"model": sam3.state_dict()}, output_weights)
    return output_weights


## SAM 3 (Vía Puntos) 

In [ ]:
import numpy as np
import os
import cv2
import pandas as pd
import torch
import time
from ultralytics import SAM

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

DATASET_ROOT   = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016_split"
WEIGHTS        = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt"
OUTPUT_WEIGHTS = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam3_isic2016.pt"
OUTPUT_PATH    = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"


def get_central_point(mask_bin):
    contours, _ = cv2.findContours(mask_bin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return [[x + w / 2, y + h / 2]]


def evaluate_model(dataset_path, weights_path):
    sam3_wrapper = SAM(WEIGHTS)
    sam3         = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sd = torch.load(weights_path)["model"]
    sam3.load_state_dict(sd)
    sam3.eval()

    iou_scores            = []
    precision_scores      = []
    recall_scores         = []
    f1_scores             = []
    dice_scores           = []
    specificity_scores    = []
    f2_scores             = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores        = []
    vram_scores           = []

    test_images_dir = os.path.join(dataset_path, "images", "test")
    test_masks_dir  = os.path.join(dataset_path, "masks",  "test")

    for img_filename in sorted(os.listdir(test_images_dir)):
        if not img_filename.lower().endswith(".jpg"):
            continue

        name      = img_filename.replace(".jpg", "")
        img_path  = os.path.join(test_images_dir, img_filename)
        mask_path = os.path.join(test_masks_dir,  name + ".png")

        if not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 127).astype(bool)
        if gt_mask.sum() == 0:
            continue

        central_point = get_central_point(gt_mask)
        if central_point is None:
            continue

        results, latency, vram = measure_inference_sam3_points(sam3_wrapper, img_path, np.array(central_point))

        if results is None or results[0].masks is None:
            continue

        scores   = results[0].boxes.conf.cpu().numpy()
        best_idx = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)
        H, W  = gt_mask.shape

        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    resultados = {
        "modelo":              ["sam3_isic2016"],
        "mean_iou":            [np.mean(iou_scores)],
        "mean_f1":             [np.mean(f1_scores)],
        "mean_recall":         [np.mean(recall_scores)],
        "mean_precision":      [np.mean(precision_scores)],
        "mean_dice":           [np.mean(dice_scores)],
        "mean_specificity":    [np.mean(specificity_scores)],
        "mean_f2":             [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":   [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":   [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":     [np.mean(latency_scores)],
        "mean_vram_mb":        [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    if os.path.exists(OUTPUT_PATH):
        df.to_csv(OUTPUT_PATH, mode="a", header=False, index=False)
    else:
        df.to_csv(OUTPUT_PATH, index=False)


trained_weights = train_sam(DATASET_ROOT, WEIGHTS, OUTPUT_WEIGHTS)
evaluate_model(DATASET_ROOT, trained_weights)

c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)


C:\Users\DanielTalavera\AppData\Local\Temp\ipykernel_896\3491141125.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(weights_path)["model"]


WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must